# 02. Cell re-segmentation with Proseg

**Why:** in the core workflow (Step 6), ~10% of cells were flagged `mixed`: their 10x cell outlines captured transcripts from neighbouring cells (segmentation spillover). Whole populations suffered: CD4 T cells were absorbed into a mixed T/endothelial cluster and dropped out of every spatial statistic, and tumour-contact T cells carried epithelial transcripts (Step 9b control row).

**What:** re-segment the tissue from the transcripts themselves with **Proseg** (Jones et al., *Nature Methods* 2025), a probabilistic method that assigns each transcript to the cell that most plausibly produced it, using the 10x nuclei as a starting point.

**How the comparison stays fair:** this notebook only produces a new segmentation and its count matrix. The core workflow then re-runs Steps 4-10 on it with identical code (a `SEGMENTATION` switch), so any difference comes from segmentation alone.

| Part | Content |
|---|---|
| A | Baseline: how good is the 10x segmentation? (metrics a new segmentation must beat) |
| B | Prepare inputs and run Proseg |
| C | Load the Proseg output and compare with the baseline |
| D | Export for the core workflow |


## Setup

Shared paths, constants and helpers come from `xenium_utils.py` at the repository root.

In [1]:
import sys
from pathlib import Path

# make the repository root (where xenium_utils.py lives) importable from notebooks/
root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "xenium_utils.py").exists())
sys.path.insert(0, str(root))
import xenium_utils as xu

xu.check_kernel()

import warnings

import numpy as np
import pandas as pd
import pyarrow.compute as pc
import pyarrow.parquet as pq
from IPython.display import display

warnings.filterwarnings("ignore", category=FutureWarning)
OUT_DIR = xu.output_dir("resegmentation")
print(f"Outputs go to {xu.display_path(OUT_DIR)}")

Outputs go to ~/data/xenium_lung/processed/resegmentation


## A. Baseline: quality of the 10x segmentation

Four numbers any re-segmentation should improve, or at least not worsen:
1. **Transcripts assigned to cells**: more is better, as long as cells stay plausible.
2. **Cells flagged `mixed`** (Step 6): direct measure of spillover.
3. **Lineage-impossible co-expression**: % of T cells with epithelial transcripts (EPCAM or MALL) and % of tumour cells with T-cell transcripts (CD3E or TRAC). In a clean segmentation both approach the background rate.
4. **Recovered populations**: whether CD4 T cells form a clean cluster.

In [2]:
# 1. Transcript assignment (gene transcripts, qv >= 20)
tx = pq.read_table(xu.OUTS_DIR / "transcripts.parquet", columns=["feature_name", "qv", "cell_id"])
is_gene = ~pc.match_substring_regex(tx["feature_name"], "^(NegControl|Unassigned|Deprecated)").to_numpy(zero_copy_only=False)
q20 = (tx["qv"].to_numpy() >= 20) & is_gene
assigned = pc.not_equal(tx["cell_id"], "UNASSIGNED").to_numpy(zero_copy_only=False)
frac_assigned = (q20 & assigned).sum() / q20.sum()
del tx

# 2-3. Spillover measures from the Step 6 annotation (all QC-passing cells)
ann = xu.load_step("annotated")
counts = ann.layers["counts"].tocsc()


def positive(genes):
    idx = [ann.var_names.get_loc(g) for g in genes]
    return np.asarray((counts[:, idx] > 0).sum(axis=1)).ravel() > 0


lineage = ann.obs["lineage"].astype(str).to_numpy()
is_t = lineage == "T / NK"
is_tumour = ann.obs["cell_type"].isin(xu.TUMOUR_TYPES).to_numpy()
baseline = pd.Series({
    "gene transcripts (qv>=20) assigned to a cell": f"{frac_assigned:.1%}",
    "cells after QC": f"{ann.n_obs:,}",
    "cells flagged 'mixed' (Step 6)": f"{(ann.obs['annotation_confidence'] == 'mixed').mean():.1%}",
    "T / NK cells with EPCAM or MALL": f"{positive(['EPCAM', 'MALL'])[is_t].mean():.1%}",
    "tumour cells with CD3E or TRAC": f"{positive(['CD3E', 'TRAC'])[is_tumour].mean():.1%}",
    "CD4 T cells, high confidence": f"{((ann.obs['cell_type'] == 'CD4 T') & (ann.obs['annotation_confidence'] == 'high')).sum():,}",
}, name="10x segmentation (baseline)")
display(baseline.to_frame())
baseline.to_csv(OUT_DIR / "baseline_10x_segmentation.csv")

,10x segmentation (baseline)
gene transcripts (qv>=20) assigned to a cell,86.6%
cells after QC,"154,394"
cells flagged 'mixed' (Step 6),9.9%
T / NK cells with EPCAM or MALL,18.8%
tumour cells with CD3E or TRAC,8.2%
"CD4 T cells, high confidence",0


## B. Run Proseg

Proseg is a command-line tool written in Rust (installed with `cargo install proseg`). Its Xenium preset reads `transcripts.parquet` directly, keeps gene transcripts with `qv >= 20`, and starts from the 10x nuclei. It then repeatedly reassigns each transcript to the cell that most plausibly produced it, letting cell shapes grow or shrink to fit the transcripts.

**Profile before running.** The full run was first tested on a 1 x 1 mm crop and its cost projected linearly. The projection matched the measured full run.

`RUN_PROSEG = False` reuses the saved run (~30 min, all CPU cores, ~13 GB RAM to redo); the run's timing is read back from its log.

In [ ]:
import re
import subprocess

RUN_PROSEG = False   # True = (re)run Proseg (~30 min, all CPU cores, up to ~13 GB RAM); False = reuse the saved run
PROSEG = Path("~/.cargo/bin/proseg").expanduser()
PROSEG_DIR = OUT_DIR / "proseg"
PROSEG_DIR.mkdir(exist_ok=True)

args = ["--xenium", str(xu.OUTS_DIR / "transcripts.parquet"),
        "--output-counts", "counts.csv.gz", "--output-cell-metadata", "cell-metadata.csv.gz",
        "--output-cell-polygons", "cell-polygons.geojson.gz", "--exclude-spatialdata-transcripts",
        "--output-spatialdata", "proseg-output.zarr", "--overwrite"]
print("Command (paths shortened):")
print("  proseg " + " ".join(xu.display_path(Path(a)) if a.startswith("/") else a for a in args), "\n")

if RUN_PROSEG or not (PROSEG_DIR / "proseg-output.zarr").exists():
    # caffeinate keeps macOS awake; /usr/bin/time -l records runtime and peak memory in run.log
    with open(PROSEG_DIR / "run.log", "w") as log:
        subprocess.run(["caffeinate", "-i", "/usr/bin/time", "-l", str(PROSEG), *args],
                       cwd=PROSEG_DIR, stdout=log, stderr=subprocess.STDOUT, check=True)

log = (PROSEG_DIR / "run.log").read_text()
grab = lambda pattern: float(re.search(pattern, log).group(1))
run = pd.Series({
    "transcripts used (qv >= 20, genes)": f"{int(grab(r'(\d+) transcripts')):,}",
    "starting cells (10x nuclei)": f"{int(grab(r'(\d+) cells')):,}",
    "wall time": f"{grab(r'([\d.]+) real') / 60:.1f} min",
    "CPU time": f"{grab(r'([\d.]+) user') / 60:.0f} min (~{grab(r'([\d.]+) user') / grab(r'([\d.]+) real'):.1f} cores busy)",
    "peak memory": f"{grab(r'(\d+)\s+peak memory footprint') / 1e9:.1f} GB",
}, name="Proseg 3.2.0, full slide")
display(run.to_frame())
print("Profiled first on a 1 x 1 mm crop (596k transcripts: 4.3 min, 0.8 GB); the linear projection for the\n"
      "full slide (~220 CPU-min, <= 14 GB) matched the measured 221 CPU-min and 12.5 GB.")

## C. Compare with the baseline

### C1. Whole-slide summary

In [ ]:
import anndata as ad
import scipy.sparse as sp
import scanpy as sc
import spatialdata as sd

# Proseg output (SpatialData: count table + cell polygons) and the 10x matrix
with warnings.catch_warnings():   # Proseg writes an older SpatialData zarr format; reading it is fine
    warnings.simplefilter("ignore", UserWarning)
    pro_sd = sd.read_zarr(PROSEG_DIR / "proseg-output.zarr")
pro = pro_sd.tables["table"]
pro_area = pro_sd.shapes["cell_boundaries"].geometry.area.to_numpy()   # 2D outline area, um^2
tenx = sc.read_10x_h5(xu.OUTS_DIR / "cell_feature_matrix.h5")            # gene features only
cells10 = pd.read_parquet(xu.OUTS_DIR / "cells.parquet").set_index("cell_id")


def summarise(adata, area):
    X = sp.csr_matrix(adata.X)
    tot = np.asarray(X.sum(axis=1)).ravel()
    genes = np.asarray((X > 0).sum(axis=1)).ravel()
    return {"cells": adata.n_obs, "cells with >= 10 transcripts": int((tot >= 10).sum()),
            "transcripts in cells": int(tot.sum()), "median transcripts / cell": float(np.median(tot)),
            "median genes / cell": float(np.median(genes)), "median cell area (um^2)": float(np.nanmedian(area))}


compare = pd.DataFrame({
    "10x segmentation": summarise(tenx, cells10.loc[tenx.obs_names, "cell_area"].to_numpy()),
    "Proseg": summarise(pro, pro_area)})
compare["Proseg / 10x"] = (compare["Proseg"] / compare["10x segmentation"]).map("{:.2f}".format)
display(compare.style.format({"10x segmentation": "{:,.0f}", "Proseg": "{:,.0f}"}))
print("Proseg starts from the 10x nuclei, so cells without a nucleus (3.5%) are not re-created; it keeps 94% of the\n"
      "transcripts in fewer, slightly larger cells, and every Proseg cell keeps its 10x id (original_cell_id).")

### C2. Paired before / after on the same cells

Because every Proseg cell keeps its 10x cell id, the comparison can follow **the same cells**, grouped by the lineage they were given in the core workflow (Step 6). For each group: the % of cells containing transcripts of a lineage they should not express.

**Finding:**
- **T / NK cells carrying epithelial transcripts drop from 18.8% to 1.5% (12.7x)**, and stromal cells carrying epithelial transcripts from 30% to 4% (7x). Tumour-to-neighbour spillover, the main problem behind the Step 6 `mixed` labels, is largely removed.
- Myeloid and epithelial cells improve about 2x.
- **T / NK cells carrying macrophage transcripts barely change (16.4% to 15.2%).** That signal may reflect genuinely tight T cell-macrophage contacts in immune hubs (or 3D overlap within the 5 µm section) rather than a 2D segmentation error.
- Among the Step 6 `mixed` groups, the CD4 T group's endothelial contamination halves (40% to 21%); the others improve by 10-15 points, so re-segmentation reduces but does not remove spillover.
- Median genes per cell fall from 29 to 24 while transcripts per cell stay at ~45: cells lose 'foreign' genes, not signal.

In [ ]:
# ---- Paired before / after: the same cells, labelled by the core workflow (Step 6) -----------
ann = xu.load_step("annotated", backed="r")
labels = ann.obs[["cell_type", "lineage", "annotation_confidence"]].copy()
pro.obs_names = pro.obs["original_cell_id"].astype(str).to_numpy()
paired = labels.index.intersection(pro.obs_names).intersection(tenx.obs_names)
print(f"{len(paired):,} cells present in both segmentations and annotated in Step 6\n")


def positive(adata, genes, cells):
    sub = adata[cells, genes].X
    return np.asarray((sp.csr_matrix(sub) > 0).sum(axis=1)).ravel() > 0


SPILL_TESTS = [   # (cell group, foreign genes it should NOT express)
    ("T / NK", ["EPCAM", "MALL"]),
    ("T / NK", ["CD163", "CD68"]),
    ("Myeloid", ["EPCAM", "MALL"]),
    ("Myeloid", ["CD3E", "TRAC"]),
    ("Epithelial", ["CD3E", "TRAC"]),
    ("Stromal", ["EPCAM", "MALL"]),
]
rows = []
for lineage, genes in SPILL_TESTS:
    cells = paired[labels.loc[paired, "lineage"].to_numpy() == lineage]
    before, after = positive(tenx, genes, cells).mean(), positive(pro, genes, cells).mean()
    rows.append({"cells (Step 6 lineage)": lineage, "foreign markers": " / ".join(genes), "n": len(cells),
                 "10x: % positive": 100 * before, "Proseg: % positive": 100 * after, "reduction": before / max(after, 1e-9)})
spill = pd.DataFrame(rows)
display(spill.style.format({"n": "{:,}", "10x: % positive": "{:.1f}", "Proseg: % positive": "{:.1f}", "reduction": "{:.1f}x"}))

# Cells flagged 'mixed' in Step 6: does Proseg clean them up?
mixed = paired[labels.loc[paired, "annotation_confidence"].to_numpy() == "mixed"]
mixed_types = labels.loc[mixed, "cell_type"].value_counts().head(6).index
mix_rows = []
for t in mixed_types:
    cells = paired[labels.loc[paired, "cell_type"].to_numpy() == t]
    partner = {"T / epithelial mixed": ["EPCAM", "MALL"], "T / myeloid mixed": ["CD163", "MS4A6A"],
               "T / fibroblast mixed": ["FBN1", "PDGFRA"], "Myeloid / epithelial mixed": ["EPCAM", "MALL"],
               "Myeloid / T mixed": ["CD3E", "TRAC"], "Myeloid / fibroblast mixed": ["FBN1", "PDGFRA"],
               "CD4 T": ["VWF", "CAVIN1"]}.get(t)
    if partner:
        mix_rows.append({"Step 6 'mixed' group": t, "n": len(cells), "contaminant genes": " / ".join(partner),
                         "10x: % positive": 100 * positive(tenx, partner, cells).mean(),
                         "Proseg: % positive": 100 * positive(pro, partner, cells).mean()})
display(pd.DataFrame(mix_rows).style.format({"n": "{:,}", "10x: % positive": "{:.1f}", "Proseg: % positive": "{:.1f}"}))
print("Each row follows the SAME cells before and after re-segmentation. Falling contaminant rates mean Proseg\n"
      "is reassigning neighbours' transcripts to the right cells, the problem behind the Step 6 'mixed' labels.")

### C3. What it looks like

A 120 µm window chosen from the data: the tile where tumour (EPCAM, MALL) and T-cell (CD3E, TRAC) transcripts meet most densely. Outlines in yellow, on DAPI.

In [ ]:
import matplotlib.pyplot as plt
import tifffile
import zarr
from matplotlib.collections import PolyCollection

# ---- Look at it: outlines and transcripts on DAPI, before vs after ---------------------------
# Choose the window from the data: the 120 um tile where tumour and T cells meet most densely
# (highest min(epithelial transcripts, T-cell transcripts)), i.e. where spillover is most likely
W = 120
border = pd.read_parquet(xu.OUTS_DIR / "transcripts.parquet", columns=["feature_name", "x_location", "y_location", "qv"],
                         filters=[("feature_name", "in", ["EPCAM", "MALL", "CD3E", "TRAC"]), ("qv", ">=", 20)])
border["kind"] = np.where(border.feature_name.isin(["EPCAM", "MALL"]), "epi", "T")
border["tile"] = list(zip(border.x_location // W, border.y_location // W))
tile_counts = border.groupby(["tile", "kind"]).size().unstack(fill_value=0)
best = tile_counts.min(axis=1).idxmax()
X0, Y0 = best[0] * W, best[1] * W
print(f"Window: x {X0:.0f}-{X0 + W:.0f} um, y {Y0:.0f}-{Y0 + W:.0f} um "
      f"({tile_counts.loc[[best], 'epi'].iloc[0]} epithelial and {tile_counts.loc[[best], 'T'].iloc[0]} T-cell transcripts)")
win = lambda xy: (xy[:, 0] > X0) & (xy[:, 0] < X0 + W) & (xy[:, 1] > Y0) & (xy[:, 1] < Y0 + W)

px = xu.PIXEL_SIZE_UM
store = tifffile.imread(xu.OUTS_DIR / "morphology_focus" / "morphology_focus_0000.ome.tif", aszarr=True, level=0)
img = zarr.open(store, mode="r")          # reads only the tiles inside the window (JPEG 2000 via imagecodecs)
r0, c0, n = int(Y0 / px), int(X0 / px), int(W / px)
dapi = np.asarray(img[0, r0:r0 + n, c0:c0 + n] if img.ndim == 3 else img[r0:r0 + n, c0:c0 + n]).astype(float)
store.close()
lo, hi = np.percentile(dapi, [1, 99.7])

tx = pd.read_parquet(xu.OUTS_DIR / "transcripts.parquet", columns=["feature_name", "x_location", "y_location", "qv"],
                     filters=[("x_location", ">", X0), ("x_location", "<", X0 + W), ("y_location", ">", Y0), ("y_location", "<", Y0 + W)])
tx = tx[tx.qv >= 20]
SHOW = {"EPCAM": "#e8743b", "MALL": "#e8743b", "CD3E": "#3fa7f5", "TRAC": "#3fa7f5"}

b10 = pd.read_parquet(xu.OUTS_DIR / "cell_boundaries.parquet")
in_win_ids = b10.loc[win(b10[["vertex_x", "vertex_y"]].to_numpy()), "cell_id"].unique()
polys10 = [g[["vertex_x", "vertex_y"]].to_numpy() for _, g in b10[b10.cell_id.isin(in_win_ids)].groupby("cell_id", sort=False)]
gdf = pro_sd.shapes["cell_boundaries"]
cent = np.column_stack([gdf.geometry.centroid.x, gdf.geometry.centroid.y])
polysP = [np.asarray(part.exterior.coords) for geom in gdf.geometry[win(cent)] for part in getattr(geom, "geoms", [geom])]

fig, axes = plt.subplots(1, 2, figsize=(14, 7))
for ax, polys, title in [(axes[0], polys10, "10x segmentation"), (axes[1], polysP, "Proseg re-segmentation")]:
    ax.imshow(np.clip((dapi - lo) / (hi - lo), 0, 1), cmap="gray", extent=[X0, X0 + W, Y0 + W, Y0])
    ax.add_collection(PolyCollection(polys, facecolors="none", edgecolors="#f5d547", linewidths=0.8))
    for gene, col in SHOW.items():
        t = tx[tx.feature_name == gene]
        ax.scatter(t.x_location, t.y_location, s=5, c=col, linewidths=0)
    ax.set_xlim(X0, X0 + W); ax.set_ylim(Y0 + W, Y0); ax.set_title(title, loc="left", fontsize=11)
    ax.set_xticks([]); ax.set_yticks([])
axes[0].plot([X0 + 5, X0 + 25], [Y0 + W - 5] * 2, color="white", lw=3)
axes[0].text(X0 + 15, Y0 + W - 8, "20 um", color="white", ha="center", fontsize=9)
fig.legend(handles=[plt.Line2D([], [], marker="o", ls="", color="#e8743b", label="epithelial (EPCAM, MALL)"),
                    plt.Line2D([], [], marker="o", ls="", color="#3fa7f5", label="T cell (CD3E, TRAC)"),
                    plt.Line2D([], [], color="#f5d547", label="cell outline")], loc="lower center", ncol=3, frameon=False)
plt.tight_layout(rect=(0, 0.05, 1, 1))
plt.show()
print("Same tissue, same transcripts. 10x outlines are fixed shapes grown from the stain images; Proseg outlines\n"
      "follow where each cell's transcripts are, so a T cell's outline stops before the neighbouring tumour cell's\n"
      "EPCAM / MALL transcripts. Proseg works on a 1 um voxel grid, hence its stepped edges.")

## D. Export for the core workflow

The Proseg counts are saved as an AnnData file in the same layout the core workflow expects: raw counts, cell area, centroids in µm, indexed by the original 10x cell ids.

**Privacy note:** Proseg stores its full command line, including local file paths, inside its output (`uns['proseg_run']`). Only the version and method are carried into the exported file.

**Next step (not yet implemented):** a `SEGMENTATION = "10x" | "proseg"` switch in the core workflow, so Steps 4-10 re-run on these cells with identical code and the effect of segmentation on the biological findings (CD4 T cells, CD8 exclusion, TLS, niches) can be measured directly.

In [ ]:
# ---- Export: Proseg counts as an AnnData the core workflow can read ------------------------
out = ad.AnnData(X=sp.csr_matrix(pro.X).astype(np.float32), var=pd.DataFrame(index=pro.var_names))
out.obs_names = pro.obs_names                          # = 10x cell ids (original_cell_id)
out.obs["cell_area"] = pro_area
out.obs["proseg_volume"] = pro.obs["volume"].to_numpy()
out.obs["transcript_counts"] = np.asarray(out.X.sum(axis=1)).ravel().astype(int)
out.obsm["spatial"] = pro.obs[["centroid_x", "centroid_y"]].to_numpy()   # um, same frame as Xenium
out.layers["counts"] = out.X.copy()
out.uns["segmentation"] = {"method": "Proseg", "version": pro.uns["proseg_run"]["version"],
                           "initialised_from": "10x nuclei", "transcript_filter": "qv >= 20, gene features"}
# note: Proseg's own uns['proseg_run'] stores the full command with local paths; it is not copied
EXPORT = OUT_DIR / f"{xu.SAMPLE}_proseg_counts.h5ad"
out.write_h5ad(EXPORT, compression="gzip")
print(f"Saved {out.n_obs:,} cells x {out.n_vars} genes to {xu.display_path(EXPORT)}")
print("Next: a SEGMENTATION = 'proseg' switch in the core workflow reads this file instead of the 10x matrix\n"
      "and re-runs Steps 4-10 with identical code, so the comparison isolates the effect of segmentation.")